In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Unit Test Runner

# COMMAND ----------

import os
import sys
import datetime

import pytest

# ---------------------------------------------------------------------------
# 1. Resolve paths dynamically.
# ---------------------------------------------------------------------------
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
print(f"Runner notebook path: {notebook_path}")

repo_root = os.path.dirname(notebook_path)
workspace_repo_root = f"/Workspace{repo_root}"

# ---------------------------------------------------------------------------
# Choose test layer: silver or gold
# ---------------------------------------------------------------------------
layer_options = ["silver", "gold"]

for layer in layer_options:
    test_file = f"{layer}_test.py"
    test_dir = os.path.join(workspace_repo_root, "", test_file)

    print(f"Resolved repo root: {workspace_repo_root}")
    print(f"Resolved test dir:  {test_dir}")

    if not os.path.isdir(test_dir):
        print(
            f"\nWARNING: test_dir does not exist: {test_dir}\n"
            f"No tests will run. Check the dirname() depth above against where "
            f"this runner notebook actually lives relative to the repo root."
        )

    os.chdir(workspace_repo_root)
    if workspace_repo_root not in sys.path:
        sys.path.insert(0, workspace_repo_root)

    sys.dont_write_bytecode = True
    os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

    run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    report_dir = os.path.join(workspace_repo_root, "")
    os.makedirs(report_dir, exist_ok=True)
    unit_report_path = f"{report_dir}/{layer}_test_{run_ts}.xml"

    # ---------------------------------------------------------------------------
    # Run pytest
    # ---------------------------------------------------------------------------
    retcode = pytest.main([
        test_dir,
        "-v",
        "-ra",
        "--import-mode=importlib",
        "-p", "no:cacheprovider",
        "--junit-xml", unit_report_path,
    ])

    print("\n" + "=" * 70)
    print("TEST SUMMARY")
    print("=" * 70)

    print("=" * 70)
    print(f"\nFull Unit_test report: {unit_report_path}")